# Stanford CS336 Llama Adaptation · Colab companion

We’re pretraining a **123,489,024-parameter Llama-style decoder from random weights** on FineWeb‑Edu. Modal’s H100 does the pretraining. This notebook does everything else on a Colab T4: it explains the math, runs the repo’s own code on a small slice of the data, reads back the Modal run, loads the finished model, samples from it, and fine-tunes it on SST‑5.

[**Open in Colab**](https://colab.research.google.com/github/xqzr2000/Stanford-CS336-Llama-Adaptation/blob/main/notebooks/CS336_Llama_Project_Walkthrough.ipynb) · [Repository](https://github.com/xqzr2000/Stanford-CS336-Llama-Adaptation)

| Chapter | What runs | Needs the Modal run? |
| --- | --- | --- |
| 1 · From token IDs to logits | The real `Llama` class, checked against the math for RMSNorm, RoPE, grouped-query attention, SwiGLU, and the loss | No |
| 2 · FineWeb‑Edu on a slice | The repo’s data preparation on a few thousand documents, including BPE training and packing | No |
| 3 · Training, small and large | The repo’s trainer on the slice with a small model, then the H100 run’s metrics and cost | Only for the H100 metrics |
| 4 · Host the final model | Loading the exported weights, the project’s sampling settings, and held-out loss | Yes |
| 5 · SST‑5 on the T4 | Five-class sentiment fine-tuning of a copy of the model | Practice mode runs without it |

**Run it.** Choose **Runtime → Change runtime type → T4 GPU**, run the setup cell, then step through with **Shift + Enter**. Setup prints the device it selected and falls back to CPU if CUDA is unavailable. Set `USE_GPU = False` to run on CPU on purpose. The notebook clones the repo at `REVISION`, which defaults to `main`, and prints the commit it used. Use a commit hash there to reproduce an earlier result.

**Before training finishes**, chapters 1–3 and practice-mode SST‑5 run on their own. Chapters 4 and 5 use the Modal run once you have copied its Colab folder to Drive. The files section after setup explains how.

## Chapter 1 · From the codebase to next-token logits

### 1 · How we prepared the codebase

| File | Job in the project |
| --- | --- |
| [`llama_lab/model.py`](https://github.com/xqzr2000/Stanford-CS336-Llama-Adaptation/blob/main/llama_lab/model.py) | Defines the configuration, decoder blocks, and language-model forward pass in PyTorch. |
| [`llama_lab/data.py`](https://github.com/xqzr2000/Stanford-CS336-Llama-Adaptation/blob/main/llama_lab/data.py) | Fetches pinned Parquet shards, trains the byte-level BPE tokenizer, and writes packed training and validation token streams. |
| [`llama_lab/train.py`](https://github.com/xqzr2000/Stanford-CS336-Llama-Adaptation/blob/main/llama_lab/train.py) | Samples windows, computes next-token loss, runs AdamW, and saves resumable checkpoints and metrics. |
| [`llama_lab/evaluate.py`](https://github.com/xqzr2000/Stanford-CS336-Llama-Adaptation/blob/main/llama_lab/evaluate.py) | Samples continuations and fine-tunes a copy of a checkpoint for five-class SST‑5 sentiment classification. |
| [`modal_app.py`](https://github.com/xqzr2000/Stanford-CS336-Llama-Adaptation/blob/main/modal_app.py) | Defines CPU data prep and sampling, H100 pretraining and sentiment jobs, and their shared Modal Volume. |
| [`pyproject.toml`](https://github.com/xqzr2000/Stanford-CS336-Llama-Adaptation/blob/main/pyproject.toml) · [`uv.lock`](https://github.com/xqzr2000/Stanford-CS336-Llama-Adaptation/blob/main/uv.lock) | Define the Python 3.12 project environment and lock its CPU and CUDA dependency sets. |
| [`.devcontainer/`](https://github.com/xqzr2000/Stanford-CS336-Llama-Adaptation/blob/main/.devcontainer/devcontainer.json) · [CI workflow](https://github.com/xqzr2000/Stanford-CS336-Llama-Adaptation/blob/main/.github/workflows/tests.yml) | Set up Codespaces and run CPU tests plus a Modal CLI help check on GitHub Actions. |
| [Plan](https://github.com/xqzr2000/Stanford-CS336-Llama-Adaptation/blob/main/docs/Plan.md) · [Runbook](https://github.com/xqzr2000/Stanford-CS336-Llama-Adaptation/blob/main/docs/Runbook.md) · [Learning log](https://github.com/xqzr2000/Stanford-CS336-Llama-Adaptation/blob/main/docs/Learning-Progress.md) · [Modal journal](https://github.com/xqzr2000/Stanford-CS336-Llama-Adaptation/blob/main/docs/cloud-walker-Modal.md) | Capture experiment decisions, commands, results, and cloud troubleshooting. |

**How it runs.** Codespaces is where we install the locked CPU environment, run tests, and submit jobs. Modal prepares data on CPU and trains on an H100. Its Volume persists `tokenizer.json`, `train.bin`, `valid.bin`, `manifest.json`, and run checkpoints across invocations. Colab uses its own PyTorch runtime and clones the repo, so this notebook calls the same functions Modal runs.

**What the data prep does.** It reads one FineWeb‑Edu Parquet shard at a time. It fits a **16,384-entry byte-level BPE tokenizer** on up to 100,000 documents from the first training shard, appends an end-of-text token after each document, and packs the IDs as unsigned 16-bit integers. The default training shards are 000–002. Validation takes the first 5 million tokens from held-out shard 013. The manifest captures the dataset revision, shard names, tokenizer fingerprint, and actual token counts. The trainer uses the manifest’s vocabulary size to build the model.

**What the tests tell us.** The repo tests the production parameter count and exercises causality, data prep, and checkpoint resume with small CPU fixtures. Live download timing, CUDA execution, and billed throughput are observations to add once those jobs run. [Pipeline tests](https://github.com/xqzr2000/Stanford-CS336-Llama-Adaptation/blob/main/tests/test_pipeline.py)

In [ ]:
USE_GPU = True  # @param {"type":"boolean"}
REVISION = "main"  # @param {"type":"string"}

from contextlib import nullcontext
from importlib.util import find_spec
from pathlib import Path
import inspect
import subprocess
import sys

assert sys.version_info >= (3, 12), "The repo needs Python 3.12 or newer. Colab provides it."
missing = [name for name in ("tokenizers", "pyarrow", "huggingface_hub", "matplotlib") if find_spec(name) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing], check=True)

import torch
from torch.nn import functional as F
from torch.nn.attention import SDPBackend, sdpa_kernel

repo_url = "https://github.com/xqzr2000/Stanford-CS336-Llama-Adaptation.git"
workspace = Path("/content") if Path("/content").is_dir() else Path.cwd()
repo = workspace / "cs336-llama-adaptation"
repo.mkdir(exist_ok=True)

def git(*args, **kwargs):
    return subprocess.run(["git", "-C", str(repo), *args], check=True, text=True, **kwargs)

if not (repo / ".git").is_dir():
    git("init", "--quiet")
assert not git("status", "--porcelain", "--", "llama_lab", capture_output=True).stdout, (
    "Restore the model source in the study checkout before running the notebook."
)
git("fetch", "--quiet", "--depth", "1", repo_url, REVISION)
git("checkout", "--quiet", "--detach", "FETCH_HEAD")
checked_out = git("rev-parse", "HEAD", capture_output=True).stdout.strip()
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

from llama_lab import data
from llama_lab.evaluate import PROMPTS, finetune_sst5, generate, load_for_eval
from llama_lab.model import Block, Llama, ModelConfig, rotate
from llama_lab.train import SEED, learning_rate, pretrain_model, precision, validate

assert Path(inspect.getfile(Llama)).resolve() == repo / "llama_lab" / "model.py", (
    "Restart the runtime if another copy of llama_lab was imported earlier."
)

torch.manual_seed(SEED)
torch.set_num_threads(2)
device = torch.device("cuda:0" if USE_GPU and torch.cuda.is_available() else "cpu")

def attention_backend(active_device):
    # Use the SDPA math backend for CUDA GQA on T4.
    return sdpa_kernel(SDPBackend.MATH) if active_device.type == "cuda" else nullcontext()

print(f"Python {sys.version.split()[0]} | PyTorch {torch.__version__}")
if device.type == "cuda":
    props = torch.cuda.get_device_properties(device)
    mode = "BF16 autocast" if isinstance(precision(device), torch.autocast) else "FP32"
    print(f"CUDA: {props.name} | {props.total_memory / 1024**3:.1f} GiB VRAM | {mode}")
elif USE_GPU:
    print("CPU | CUDA unavailable. To try the T4, change the Colab runtime type.")
else:
    print("CPU | selected in setup")
print(f"Repo: {REVISION} at commit {checked_out[:7]}")

### Your files from Modal

Chapters 4 and 5 use the finished Modal run. After pretraining, run the export from the repo root in Codespaces. It writes one `colab` folder with FP16 weights (about 250 MB), `tokenizer.json`, `manifest.json`, the 10 MB `valid.bin`, and `metrics.json` with every checkpoint’s metrics. Then download that folder:

```bash
m run modal_app.py::export --run-id llama124m
m volume get cs336-llama-adaptation-artifacts runs/llama124m/colab llama124m-colab
```

Upload the `llama124m-colab` folder to Google Drive. Set `MODEL_FOLDER` below to its path under `/content/drive/MyDrive/` and set `MOUNT_DRIVE = True`. A folder you uploaded to the runtime also works, for example `/content/llama124m-colab`. A training checkpoint, `latest.pt`, can stand in for the weights file; it is about 1.48 GB because it also holds AdamW state.

You can export the pilot run (`--run-id pilot`) to try the loading and sampling cells early. Its generations reflect how little training it has seen.

In [ ]:
MODEL_FOLDER = ""  # @param {"type":"string"}
MOUNT_DRIVE = False  # @param {"type":"boolean"}

model_folder = Path(MODEL_FOLDER.strip()) if MODEL_FOLDER.strip() else None
if model_folder and MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

def find_weights(folder):
    """Prefer the newest FP16 weights export, then a full training checkpoint."""
    exports = sorted(folder.glob("weights-*.pt"))
    if exports:
        return exports[-1]
    if (folder / "latest.pt").is_file():
        return folder / "latest.pt"
    raise FileNotFoundError(f"No weights-*.pt or latest.pt in {folder}")

if model_folder is None:
    print("MODEL_FOLDER is blank. Chapters 4 and 5 will skip the final model.")
else:
    for path in sorted(model_folder.iterdir()):
        print(f"{path.name:>24}  {path.stat().st_size / 1e6:9.1f} MB")
    print("Weights:", find_weights(model_folder).name)

### 2 · Read the configuration and count the weights

A decoder-only transformer reads a prefix and predicts what comes next. We use $B$ for batch size, $T$ for sequence length, $D$ for model width, $V$ for vocabulary size, and $d_h$ for head width.

| Setting in `ModelConfig` | Production | Small example |
| --- | ---: | ---: |
| Vocabulary $V$ | 16,384 | 64 |
| Width $D$ | 768 | 48 |
| Decoder blocks $L$ | 12 | 2 |
| Query / key-value heads | 12 / 4 | 6 / 2 |
| Head width $d_h=D/H_q$ | 64 | 8 |
| SwiGLU hidden width $M$ | 3,328 | 96 |
| Context limit | 256 | 16 |

Each block has $2D^2+2D(H_{kv}d_h)$ attention weights across Q, K, V, and output projections, $3DM$ feed-forward weights, and $2D$ RMSNorm weights. The full model adds $VD$ for the shared input embedding/output matrix and $D$ for the final RMSNorm. RoPE sine and cosine tables are buffers computed from the configuration. The projections have no biases.

$$
N = VD + L\big[2D^2 + 2D(H_{kv}d_h) + 3DM + 2D\big] + D.
$$

The first model below is created on PyTorch’s `meta` device, which records parameter shapes without allocating the full weights. The small model uses the same `Llama` class with FP32 weights on the device printed by setup. [`ModelConfig` and `Llama`](https://github.com/xqzr2000/Stanford-CS336-Llama-Adaptation/blob/main/llama_lab/model.py)

In [ ]:
production = ModelConfig()

def parameter_count(config):
    width = config.width
    head_width = width // config.query_heads
    kv_width = config.kv_heads * head_width
    attention = 2 * width**2 + 2 * width * kv_width
    feed_forward = 3 * width * config.hidden_width
    block = attention + feed_forward + 2 * width
    return config.vocab_size * width + config.layers * block + width

with torch.device("meta"):
    model_spec = Llama(production)
counted = sum(parameter.numel() for parameter in model_spec.parameters())
assert counted == parameter_count(production) == 123_489_024
print(f"Production weights: {counted:,}")

config = ModelConfig(
    vocab_size=64, width=48, layers=2, query_heads=6, kv_heads=2,
    hidden_width=96, context=16,
)
model = Llama(config).to(device).eval()
assert sum(p.numel() for p in model.parameters()) == parameter_count(config)
print(f"Small example weights: {parameter_count(config):,} on {next(model.parameters()).device}")

### 3 · Follow one batch through the forward pass

The input is a matrix of integer IDs, `[B, T]`. `nn.Embedding` looks up one width-$D$ vector per ID. The sequence of vectors stays `[B, T, D]` through each decoder block and the final RMSNorm. In `forward`, `F.linear` multiplies each position’s features by the transposed embedding matrix to produce `[B, T, V]` logits. Sharing that matrix is called **weight tying**.

| Boundary | Small example | Production at the default training batch |
| --- | --- | --- |
| Token IDs | `[2, 5]` | `[64, 256]` |
| Embeddings and block output | `[2, 5, 48]` | `[64, 256, 768]` |
| Queries | `[2, 6, 5, 8]` | `[64, 12, 256, 64]` |
| Keys and values, each | `[2, 2, 5, 8]` | `[64, 4, 256, 64]` |
| SwiGLU intermediate | `[2, 5, 96]` | `[64, 256, 3328]` |
| Next-token logits | `[2, 5, 64]` | `[64, 256, 16384]` |

The logits at position $t$ use the prefix ending at $t$ and score the token at $t+1$. The following cell captures the real layer outputs with temporary PyTorch hooks. [`features` and `forward`](https://github.com/xqzr2000/Stanford-CS336-Llama-Adaptation/blob/main/llama_lab/model.py#L107)

In [ ]:
# Each row is a six-token window. Inputs and targets are shifted by one.
packed = torch.tensor([[4, 11, 9, 21, 6, 3], [8, 5, 17, 2, 10, 7]], device=device)
inputs = packed[:, :-1].contiguous()
targets = packed[:, 1:].contiguous()

def record_shape(name, shapes):
    def hook(module, args, output):
        shapes[name] = tuple(output.shape)
    return hook

shapes = {"token IDs": tuple(inputs.shape)}
layers = {
    "embedding": model.embedding,
    **{f"block {i}": block for i, block in enumerate(model.blocks)},
    "final RMSNorm": model.norm,
}
handles = [layer.register_forward_hook(record_shape(name, shapes)) for name, layer in layers.items()]
try:
    with torch.no_grad(), attention_backend(device):
        logits = model(inputs)
finally:
    for handle in handles:
        handle.remove()

shapes["logits"] = tuple(logits.shape)
for name, shape in shapes.items():
    print(f"{name:>14}  {shape}")
assert logits.shape == (*inputs.shape, config.vocab_size)
print("First row input: ", inputs[0].tolist())
print("First row target:", targets[0].tolist())

### 4 · Read the block from top to bottom

For a token vector $x\in\mathbb{R}^D$, RMSNorm rescales by the root mean square across its features and learns one scale $\gamma_i$ per feature.

$$
\operatorname{RMSNorm}(x)_i
= \gamma_i\,\frac{x_i}{\sqrt{\frac{1}{D}\sum_{j=1}^{D}x_j^2+\epsilon}},
\qquad \epsilon=10^{-5}.
$$

The mean here is the mean of squared values. The code computes this in float32 and returns the input dtype. The block applies normalization **before** attention and before the feed-forward sublayer. Each sublayer adds a residual update back to the stream.

$$
u=x+\operatorname{Attention}(\operatorname{RMSNorm}_{\mathrm{attn}}(x)),\qquad
y=u+\operatorname{FFN}(\operatorname{RMSNorm}_{\mathrm{ffn}}(u)).
$$

Read `Block.forward` and look for the two additions. The first mixes information across token positions. The second transforms features separately at each token position. The model also uses a smaller initialization standard deviation for the two projections that return residual updates. [`RMSNorm`](https://github.com/xqzr2000/Stanford-CS336-Llama-Adaptation/blob/main/llama_lab/model.py#L27) · [`Block`](https://github.com/xqzr2000/Stanford-CS336-Llama-Adaptation/blob/main/llama_lab/model.py#L68) · [Initialization](https://github.com/xqzr2000/Stanford-CS336-Llama-Adaptation/blob/main/llama_lab/model.py#L96)

In [ ]:
print(inspect.getsource(Block.forward))

first_block = model.blocks[0]
with torch.no_grad():
    embedded = model.embedding(inputs)
    normalized = first_block.attn_norm(embedded)
    expected_norm = embedded * torch.rsqrt(embedded.square().mean(dim=-1, keepdim=True) + 1e-5)
    expected_norm = expected_norm * first_block.attn_norm.weight
torch.testing.assert_close(normalized, expected_norm)
print("RMSNorm matches the equation.")

### 5 · RoPE encodes relative position in Q–K scores

The code pairs the two halves of each head vector. For head width $d_h=8$, the pairs are $(0,4),(1,5),(2,6),(3,7)$. Pair $i$ rotates by angle $\theta_{p,i}=p\omega_i$ at token position $p$, where $\omega_i=10000^{-2i/d_h}$. For a pair $(a,b)$,

$$
R(\theta)\begin{bmatrix}a\\ b\end{bmatrix}
=
\begin{bmatrix}
a\cos\theta-b\sin\theta\\
a\sin\theta+b\cos\theta
\end{bmatrix}.
$$

The code uses this rotation on **queries and keys**. For a query at position $p$ and a key at position $s$, one coordinate pair contributes

$$
\big(R(p\omega_i)q_i\big)^\top R(s\omega_i)k_i
= q_i^\top R(-p\omega_i)R(s\omega_i)k_i
= q_i^\top R((s-p)\omega_i)k_i.
$$

The first equality uses $R(\theta)^\top=R(-\theta)$. The second uses composition of rotations. Summing over the pairs gives the full dot product. For fixed query and key vectors, moving both positions by the same amount preserves that dot product. The position-dependent rotation in the score therefore depends on the offset $s-p$.

`Llama.__init__` precomputes the angles for each position and pair. `rotate` splits the last dimension in two and applies the equation elementwise. Values travel through attention without this rotation. The small numerical identity check below uses float64 on CPU. [`rotate`](https://github.com/xqzr2000/Stanford-CS336-Llama-Adaptation/blob/main/llama_lab/model.py#L38) · [RoPE tables](https://github.com/xqzr2000/Stanford-CS336-Llama-Adaptation/blob/main/llama_lab/model.py#L91) · [RoFormer paper](https://arxiv.org/abs/2104.09864)

In [ ]:
print(inspect.getsource(rotate))

# At pi/2, (1, 0) rotates to (0, 1).
pair = torch.tensor([1.0, 0.0])
quarter_turn = torch.tensor([torch.pi / 2])
rotated = rotate(pair, quarter_turn.cos(), quarter_turn.sin())
torch.testing.assert_close(rotated, torch.tensor([0.0, 1.0]), atol=1e-6, rtol=0)

head_width = config.width // config.query_heads
query_vector = torch.linspace(-0.6, 0.8, head_width, dtype=torch.float64)
key_vector = torch.linspace(0.3, -0.4, head_width, dtype=torch.float64)
frequencies = 10_000.0 ** (-torch.arange(0, head_width, 2, dtype=torch.float64) / head_width)

def rotate_at(vector, position):
    angle = position * frequencies
    return rotate(vector, angle.cos(), angle.sin())

p, s = 5, 2
score = rotate_at(query_vector, p) @ rotate_at(key_vector, s)
relative_score = query_vector @ rotate_at(key_vector, s - p)
shifted_score = rotate_at(query_vector, p + 4) @ rotate_at(key_vector, s + 4)
torch.testing.assert_close(score, relative_score)
torch.testing.assert_close(score, shifted_score)
torch.testing.assert_close(rotate_at(query_vector, p).norm(), query_vector.norm())
print(f"Q·K at ({p}, {s}): {score.item():.6f}")
print(f"Q·K at ({p + 4}, {s + 4}): {shifted_score.item():.6f}")

### 6 · Grouped-query attention shares K and V across query heads

`Attention.forward` projects the same normalized stream into Q, K, and V, then reshapes and transposes to `[B, heads, T, head_width]`. Production has $H_q=12$ and $H_{kv}=4$, so **three query heads share each key head and its matching value head**. For zero-based query head $h$, its key/value group is $g(h)=\lfloor h/3\rfloor$. The small model also has three query heads per group.

For query position $t$, allowed key positions are $s\le t$. With $\widetilde q,\widetilde k$ denoting the RoPE-transformed vectors,

$$
\ell_{h,t,s}=\frac{\widetilde q_{h,t}^{\top}\widetilde k_{g(h),s}}{\sqrt{d_h}}
+\begin{cases}0 & s\le t\\ -\infty & s>t,\end{cases}
\qquad
a_{h,t,s}=\operatorname{softmax}_{s}(\ell_{h,t,s}),
\qquad
z_{h,t}=\sum_s a_{h,t,s}v_{g(h),s}.
$$

If the components have zero mean and unit variance and are independent, the unscaled dot product has variance $d_h$. Dividing by $\sqrt{d_h}$ keeps score variance around one in that idealized calculation. Each query head still gets its own weights $a_{h,t,s}$ because each has its own queries.

The K and V projections each map $768\to256$ in production. With one K and V head per query head, they would each map $768\to768$. GQA saves **786,432 projection weights per block** here. The code uses `scaled_dot_product_attention(..., is_causal=True, enable_gqa=True)` for the attention computation, then joins the query heads and applies `out`. On CUDA, this notebook selects PyTorch’s math backend for GQA and uses FP32 throughout. The repo’s H100 training path uses BF16 autocast. [`Attention.forward`](https://github.com/xqzr2000/Stanford-CS336-Llama-Adaptation/blob/main/llama_lab/model.py#L55) · [PyTorch SDPA](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html) · [GQA paper](https://arxiv.org/abs/2305.13245)

In [ ]:
attention = first_block.attention
batch_size, length, width = normalized.shape
cos = model.rope_cos[:length]
sin = model.rope_sin[:length]

def split_heads(projected, heads):
    return projected.reshape(batch_size, length, heads, head_width).transpose(1, 2)

with torch.no_grad(), attention_backend(device):
    queries = split_heads(attention.q(normalized), config.query_heads)
    keys = split_heads(attention.k(normalized), config.kv_heads)
    values = split_heads(attention.v(normalized), config.kv_heads)
    q_rot = rotate(queries, cos, sin)
    k_rot = rotate(keys, cos, sin)

    heads_per_group = config.query_heads // config.kv_heads
    group = torch.arange(config.query_heads, device=device) // heads_per_group
    scores = q_rot @ k_rot[:, group].transpose(-2, -1) / head_width**0.5
    future = torch.ones(length, length, dtype=torch.bool, device=device).triu(1)
    weights = scores.masked_fill(future, -torch.inf).softmax(dim=-1)
    reference_heads = weights @ values[:, group]
    reference = attention.out(reference_heads.transpose(1, 2).reshape(batch_size, length, width))

    actual_attention = attention(normalized, cos, sin)

torch.testing.assert_close(actual_attention, reference, atol=1e-6, rtol=1e-5)
assert torch.count_nonzero(weights[..., future]) == 0
print("Q / K / V shapes:", tuple(queries.shape), tuple(keys.shape), tuple(values.shape))
print("Query head → KV head:", group.tolist())
print("Allowed attention weights for query head 0 at position 2:")
print(weights[0, 0, 2].round(decimals=4))
print("Manual grouped attention matches the repository.")

### 7 · SwiGLU learns which features to pass through

The feed-forward sublayer acts on one normalized token vector $h\in\mathbb{R}^D$ at a time. PyTorch stores the `gate` and `up` matrices as $[M,D]$, and the `down` matrix as $[D,M]$. Using column vectors,

$$
g=W_{\mathrm{gate}}h,\qquad u=W_{\mathrm{up}}h,\qquad
\operatorname{SiLU}(g)=g\odot\sigma(g),\qquad
\operatorname{FFN}(h)=W_{\mathrm{down}}\big(\operatorname{SiLU}(g)\odot u\big),
$$

where $\sigma(g)=1/(1+e^{-g})$ elementwise and $\odot$ denotes elementwise multiplication. In hidden coordinate $i$, the product is $g_i\sigma(g_i)u_i$. When $g_i=0$ the coordinate closes. A large positive $g_i$ can amplify $u_i$. SiLU can also take negative values. The `down` projection mixes the resulting $M$ features into a width-$D$ residual update. Production does $768\to3328\to768$ at every token position.

Find `F.silu(self.gate(h)) * self.up(h)` in `Block.forward`. Both input projections read the same normalized vector. The multiplication is the gating operation. [`Block.forward`](https://github.com/xqzr2000/Stanford-CS336-Llama-Adaptation/blob/main/llama_lab/model.py#L78) · [GLU Variants Improve Transformer](https://arxiv.org/abs/2002.05202)

In [ ]:
with torch.no_grad(), attention_backend(device):
    after_attention = embedded + actual_attention
    h = first_block.ffn_norm(after_attention)
    gate = first_block.gate(h)
    up = first_block.up(h)
    gated = (gate * gate.sigmoid()) * up
    update = first_block.down(gated)
    reference_block = after_attention + update
    actual_block = first_block(embedded, cos, sin)

torch.testing.assert_close(reference_block, actual_block, atol=1e-6, rtol=1e-5)
print("gate / up / gated:", tuple(gate.shape), tuple(up.shape), tuple(gated.shape))
print("residual update: ", tuple(update.shape))
print("Manual SwiGLU and both residual paths match the repository.")

### 8 · From logits to the training objective

Given a window of $T+1$ IDs, the trainer uses the first $T$ as inputs and the last $T$ as targets. For example, input `[4, 11, 9]` pairs with target `[11, 9, 21]`. The last dimension of the logits is the vocabulary axis. The loss is the average negative log probability of the actual next token:

$$
\mathcal{L}=-\frac{1}{BT}\sum_{b,t}
\log\left(\operatorname{softmax}(z_{b,t,:})_{y_{b,t}}\right).
$$

The training code flattens the first two dimensions to get `[B*T, V]` logits and `[B*T]` targets. PyTorch’s cross-entropy takes raw logits. During backpropagation the shared embedding matrix gets gradient through both the input lookup and the output projection.

The cell below also changes the last input token and compares earlier logits. Earlier positions should stay the same because the causal mask excludes that future token. [Training windows and loss](https://github.com/xqzr2000/Stanford-CS336-Llama-Adaptation/blob/main/llama_lab/train.py#L29) · [Causality test](https://github.com/xqzr2000/Stanford-CS336-Llama-Adaptation/blob/main/tests/test_pipeline.py)

In [ ]:
with torch.no_grad(), attention_backend(device):
    edited = inputs.clone()
    edited[0, -1] = (edited[0, -1] + 1) % config.vocab_size
    edited_logits = model(edited)
    torch.testing.assert_close(logits[0, :-1], edited_logits[0, :-1], atol=1e-6, rtol=1e-5)
    assert not torch.allclose(logits[0, -1], edited_logits[0, -1])

model.zero_grad(set_to_none=True)
with attention_backend(device):
    features = model.features(inputs)
manual_logits = features @ model.embedding.weight.T
torch.testing.assert_close(manual_logits, logits, atol=1e-6, rtol=1e-5)

loss = F.cross_entropy(manual_logits.flatten(0, 1), targets.flatten())
target_log_probs = manual_logits.log_softmax(dim=-1).gather(-1, targets.unsqueeze(-1))
torch.testing.assert_close(loss, -target_log_probs.mean())
with attention_backend(device):
    loss.backward()
grad = model.embedding.weight.grad
assert grad is not None and torch.isfinite(grad).all() and grad.norm() > 0

print(f"Synthetic batch cross-entropy: {loss.item():.3f} nats/token")
print(f"Shared embedding gradient norm: {grad.norm().item():.3f}")
print("Changing the final token left all earlier logits unchanged.")

### 9 · Compare the same forward pass on CPU and T4

This demo copies the same small model and input to both devices, warms up each one, and times the same repeated forward pass. The CUDA synchronization calls wait for pending GPU work at the start and end of timing. The input is already on the device, so the measurement excludes data transfer. Small batches and Python overhead can dominate this demo. Treat it as a lesson in measurement, and use the Modal pilot to measure H100 training throughput.

On a T4, check how closely the logits match before comparing timing. CUDA and CPU kernels can round floating-point operations differently. [PyTorch CUDA timing notes](https://docs.pytorch.org/docs/stable/notes/cuda.html#asynchronous-execution)

In [ ]:
from copy import deepcopy
from time import perf_counter

cpu_device = torch.device("cpu")
cpu_model = deepcopy(model).to(cpu_device).eval()
cpu_batch = inputs.cpu().repeat(16, 1)

@torch.inference_mode()
def timed_forward(network, batch, warmup=5, repeats=30):
    active_device = batch.device
    with attention_backend(active_device):
        for _ in range(warmup):
            network(batch)
        if active_device.type == "cuda":
            torch.cuda.synchronize(active_device)
        start = perf_counter()
        for _ in range(repeats):
            result = network(batch)
        if active_device.type == "cuda":
            torch.cuda.synchronize(active_device)
        elapsed = perf_counter() - start
    return result.cpu(), 1_000 * elapsed / repeats

print(f"Timed input: {tuple(cpu_batch.shape)} token IDs")
cpu_logits, cpu_ms = timed_forward(cpu_model, cpu_batch)
print(f"CPU: {cpu_ms:.3f} ms per forward")
if device.type == "cuda":
    gpu_logits, gpu_ms = timed_forward(model, cpu_batch.to(device))
    torch.testing.assert_close(gpu_logits, cpu_logits, atol=1e-4, rtol=1e-3)
    print(f"{torch.cuda.get_device_name(device)}: {gpu_ms:.3f} ms per forward")
    print(f"Max absolute logit difference: {(gpu_logits - cpu_logits).abs().max().item():.2e}")
else:
    print("T4 comparison appears when the selected device is CUDA.")

### 10 · Code review notes for this project

- **Causal scope.** A 256-token context bounds what the model can attend to in one forward pass. When generation runs longer, `generate` keeps only the most recent context window and recomputes the forward pass. This recomputes the prefix’s keys and values at every generated token. For the first samples, look for continuity across a short explanation. [Generation loop](https://github.com/xqzr2000/Stanford-CS336-Llama-Adaptation/blob/main/llama_lab/evaluate.py#L35)
- **Document boundaries.** Data prep inserts an end-of-text ID between documents. The trainer samples windows from the packed stream, so a window can cross an end-of-text marker and attention can see earlier positions across that boundary. The boundary marker has its own learned embedding. [Packing](https://github.com/xqzr2000/Stanford-CS336-Llama-Adaptation/blob/main/llama_lab/data.py#L80) · [Window sampling](https://github.com/xqzr2000/Stanford-CS336-Llama-Adaptation/blob/main/llama_lab/train.py#L137)
- **Validation.** The trainer evaluates up to eight batches of evenly spaced windows from the held-out stream. With the default batch size that is 512 windows, or 131,072 scored next-token positions. Keep the tokenizer, held-out data, and validation batch setting fixed when comparing checkpoints. [Validation function](https://github.com/xqzr2000/Stanford-CS336-Llama-Adaptation/blob/main/llama_lab/train.py#L54)
- **Educational continuations.** Keep the same prompts and decoding settings across checkpoints. For each continuation, record whether it stays on topic, develops an explanation, and gets the basic facts right. Use held-out loss alongside these observations. [Default prompts](https://github.com/xqzr2000/Stanford-CS336-Llama-Adaptation/blob/main/llama_lab/evaluate.py#L17) · [Project plan](https://github.com/xqzr2000/Stanford-CS336-Llama-Adaptation/blob/main/docs/Plan.md)

## Chapter 2 · FineWeb‑Edu on a slice

Modal prepares about 2.5B training tokens. Here we run **the same `prepare_data` function** on the first few thousand documents of the same pinned shards, so every step can be inspected in a few minutes.

**Byte-level BPE.** Text is first turned into UTF‑8 bytes, so the starting vocabulary is the 256 possible byte values and no input is ever out of vocabulary. Training counts adjacent symbol pairs inside pre-tokenized chunks (words with their leading space), merges the most frequent pair into a new symbol, and repeats. The final vocabulary is

$$
V = 256\ \text{bytes} + 2\ \text{special tokens} + \text{merges},
$$

so the production tokenizer learns $16{,}384-258=16{,}126$ merges. The special tokens are `<|endoftext|>` and `<|pad|>`.

**Packing.** Each document’s IDs are followed by one end-of-text ID, and all documents are concatenated into one stream. With $V\le 65{,}536$ every ID fits in an unsigned 16-bit integer, so a token costs 2 bytes on disk and `train.bin` holds $2N$ bytes for $N$ tokens.

**Windows.** The trainer draws a start $s$ uniformly and reads $T+1$ consecutive IDs. Inputs are positions $s\ldots s+T-1$ and targets are $s+1\ldots s+T$.

**Reading tokens.** In printed tokens, `Ġ` stands for a leading space and `Ċ` for a newline.

**Compression.** Bytes per token measures how much text one ID covers. A larger vocabulary covers more bytes per token, so the same text costs fewer training tokens.

The slice reads only the first documents of shard 000 for training and shard 013 for validation, through Hugging Face’s file system at the revision the repo pins. It trains a smaller 4,096-entry tokenizer so BPE has enough text for every merge. [`prepare_data`](https://github.com/xqzr2000/Stanford-CS336-Llama-Adaptation/blob/main/llama_lab/data.py#L104) · [`train_tokenizer`](https://github.com/xqzr2000/Stanford-CS336-Llama-Adaptation/blob/main/llama_lab/data.py#L66) · [`encode_texts`](https://github.com/xqzr2000/Stanford-CS336-Llama-Adaptation/blob/main/llama_lab/data.py#L80)

In [ ]:
SLICE_DOCUMENTS = 2000  # @param {"type":"integer"}
SLICE_VOCAB = 4096  # @param {"type":"integer"}
SLICE_SOURCE = ""  # @param {"type":"string"}

import shutil
from itertools import islice

import pyarrow as pa
import pyarrow.parquet as pq

slice_root = workspace / "fineweb-slice"
shutil.rmtree(slice_root, ignore_errors=True)

def open_shard(name):
    """Stream a pinned shard from Hugging Face, or read a local copy named in SLICE_SOURCE."""
    if SLICE_SOURCE.strip():
        return open(Path(SLICE_SOURCE.strip()) / name, "rb")
    from huggingface_hub import HfFileSystem

    remote = f"datasets/{data.DATASET}@{data.REVISION}/{data.SUBSET}/{name}"
    return HfFileSystem().open(remote, "rb", block_size=8 << 20)

def fetch_slice(name, path):
    count = SLICE_DOCUMENTS if name != data.VALID_SHARD else max(SLICE_DOCUMENTS // 10, 100)
    with open_shard(name) as source:
        texts = list(islice(data.parquet_texts(source), count))
    pq.write_table(pa.table({"text": texts}), path)

slice_manifest = data.prepare_data(
    slice_root, SLICE_VOCAB, 1, scratch=workspace / "fineweb-scratch", fetch=fetch_slice,
)
print({key: slice_manifest[key] for key in ("dataset", "revision", "files", "vocab_size", "tokens")})

In [ ]:
import json

import numpy as np
from tokenizers import Tokenizer

slice_tokenizer = Tokenizer.from_file(str(slice_root / "tokenizer.json"))
eos = slice_tokenizer.token_to_id(data.END)
merges = json.loads((slice_root / "tokenizer.json").read_text())["model"]["merges"]
print(f"Merges learned: {len(merges):,} = {SLICE_VOCAB:,} − 256 bytes − 2 special tokens")
print("First merges:", [" + ".join(pair) if isinstance(pair, list) else pair for pair in merges[:8]])

sentence = "Photosynthesis converts light energy into chemical energy."
pieces = slice_tokenizer.encode(sentence).tokens
print(f"\n{len(pieces)} tokens:", pieces)
assert slice_tokenizer.decode(slice_tokenizer.encode(sentence).ids) == sentence

stream = np.fromfile(slice_root / "train.bin", dtype="<u2")
assert (slice_root / "train.bin").stat().st_size == 2 * len(stream)
print(f"\ntrain.bin: {len(stream):,} tokens, {(stream == eos).sum():,} documents, {stream.nbytes:,} bytes")
first_end = int(np.flatnonzero(stream == eos)[0])
print("End of the first document:", repr(slice_tokenizer.decode(stream[max(0, first_end - 12):first_end].tolist())))

with open_shard(data.VALID_SHARD) as source:
    sample_texts = list(islice(data.parquet_texts(source), 200))
utf8_bytes = sum(len(text.encode()) for text in sample_texts)

def bytes_per_token(tokenizer):
    return utf8_bytes / sum(len(e.ids) for e in tokenizer.encode_batch(sample_texts))

print(f"\nBytes per token on 200 validation documents, {SLICE_VOCAB:,}-entry slice tokenizer: {bytes_per_token(slice_tokenizer):.2f}")
if model_folder and (model_folder / "tokenizer.json").is_file():
    production_tokenizer = Tokenizer.from_file(str(model_folder / "tokenizer.json"))
    print(f"Bytes per token, the run's {production_tokenizer.get_vocab_size():,}-entry tokenizer: "
          f"{bytes_per_token(production_tokenizer):.2f}")
else:
    print("Set MODEL_FOLDER to compare with the run's 16,384-entry tokenizer.")

## Chapter 3 · Training, small and large

**Objective.** Training minimizes the next-token cross-entropy from §8. Perplexity is $e^{\mathcal{L}}$. A randomly initialized model predicts roughly uniformly, so its first loss is near $\ln V$: about 9.70 for $V=16{,}384$.

**Optimizer.** AdamW keeps running averages of the gradient and of its square, with $\beta_1=0.9$ and $\beta_2=0.95$, and applies weight decay $0.1$ to every parameter. Two FP32 moment tensors per weight are why a full checkpoint is three times the size of the weights.

**Learning-rate schedule.** For optimizer step $s$, warmup $W$, schedule length $S$ and peak $\eta_{\max}$,

$$
\eta(s)=
\begin{cases}
\eta_{\max}\,\dfrac{s}{W} & s\le W\\[6pt]
\eta_{\max}\left[0.1+0.9\cdot\dfrac{1+\cos\!\big(\pi\,\frac{s-W}{S-W}\big)}{2}\right] & s>W,
\end{cases}
$$

a linear warmup followed by a cosine decay to 10% of the peak. The schedule length is stored in the checkpoint, which is why the main run fixes it before its first step.

**Tokens and cost.** Each optimizer step processes $B\times A\times T$ tokens, $64\times1\times256=16{,}384$ in production. The H100 time for $S$ steps at measured throughput $r$ tokens per second is $S\cdot16{,}384/r$ seconds, priced at about US$4.33 per hour in the [plan’s cost estimate](https://github.com/xqzr2000/Stanford-CS336-Llama-Adaptation/blob/main/docs/Plan.md#cost-estimate).

The next cells plot the production schedule, run **the repo’s `pretrain_model`** on the slice with a small model, stop and resume it from its checkpoint, then read back the H100 run if `MODEL_FOLDER` holds its metrics. [`learning_rate`](https://github.com/xqzr2000/Stanford-CS336-Llama-Adaptation/blob/main/llama_lab/train.py#L42) · [`pretrain_model`](https://github.com/xqzr2000/Stanford-CS336-Llama-Adaptation/blob/main/llama_lab/train.py#L71)

In [ ]:
import matplotlib.pyplot as plt

S, W, PEAK = 150_000, 200, 3e-4
steps = np.unique(np.concatenate([np.arange(1, 2_001), np.linspace(1, S, 2_000).astype(int)]))
rates = [learning_rate(int(s), S, W, PEAK) for s in steps]
for s in (1, W, S // 2, S):
    print(f"step {s:>7,}: lr = {learning_rate(s, S, W, PEAK):.2e}")

fig, ax = plt.subplots(figsize=(7, 2.8))
ax.plot(steps, rates)
ax.set(xlabel="optimizer step", ylabel="learning rate", title="Production schedule (defaults)")
ax.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
plt.show()

In [ ]:
PRACTICE_STEPS = 300  # @param {"type":"integer"}

practice_root = workspace / "practice-run"
shutil.rmtree(practice_root, ignore_errors=True)
practice_config = ModelConfig(
    vocab_size=SLICE_VOCAB, width=128, layers=2, query_heads=4, kv_heads=2,
    hidden_width=344, context=64,
)
practice = dict(
    schedule_steps=PRACTICE_STEPS, warmup=20, micro_batch=16, accumulation=1, peak_lr=3e-3,
    minutes=10, checkpoint_every=100, log_every=50,
)
print(f"ln V = {np.log(SLICE_VOCAB):.2f}")
halfway = PRACTICE_STEPS // 2
with attention_backend(device):
    pretrain_model(practice_config, slice_root, practice_root, slice_manifest, device,
                   target_step=halfway, **practice)
    print("--- stopped; a new call resumes from latest.pt ---")
    pretrain_model(practice_config, slice_root, practice_root, slice_manifest, device,
                   target_step=PRACTICE_STEPS, **practice)

practice_metrics = [json.loads(p.read_text()) for p in sorted(practice_root.glob("metrics-*.json"))]
for row in practice_metrics:
    print(f"step {row['step']:>4}: val_loss {row['val_loss']:.3f}, train_loss {row['train_loss']:.3f}")

In [ ]:
H100_DOLLARS_PER_HOUR = 4.33  # @param {"type":"number"}

metrics_path = model_folder / "metrics.json" if model_folder else None
if metrics_path is None or not metrics_path.is_file():
    print("Set MODEL_FOLDER to a Colab folder from the export to read the H100 run.")
else:
    run = json.loads(metrics_path.read_text())
    rows = run["checkpoints"]
    print(f"Run {run['run_id']}: {len(rows)} checkpoints")
    print(f"{'step':>8} {'tokens':>14} {'val_loss':>9} {'ppl':>9} {'tokens/s':>10} {'val+save s':>11}")
    for row in rows:
        print(f"{row['step']:>8,} {row['tokens']:>14,} {row['val_loss']:>9.3f} {row['perplexity']:>9.1f} "
              f"{row.get('train_tokens_per_second', float('nan')):>10,.0f} "
              f"{row.get('validation_and_save_seconds', float('nan')):>11.1f}")
    speeds = [row["train_tokens_per_second"] for row in rows if "train_tokens_per_second" in row]
    if speeds:
        rate = float(np.median(speeds))
        hours = rows[-1]["tokens"] / rate / 3600
        print(f"\nMedian throughput {rate:,.0f} tokens/s → {hours:.1f} H100 hours for "
              f"{rows[-1]['tokens']:,} tokens ≈ ${hours * H100_DOLLARS_PER_HOUR:,.2f} of training steps")
    fig, ax = plt.subplots(figsize=(7, 2.8))
    ax.plot([row["tokens"] for row in rows], [row["val_loss"] for row in rows], marker=".")
    ax.set(xscale="log", xlabel="processed tokens", ylabel="validation loss", title=f"H100 run {run['run_id']}")
    plt.show()

## Chapter 4 · Host the final model

**Weights on disk.** The export stores each weight in FP16, two bytes per parameter, so 123,489,024 parameters take about 247 MB. Loading converts them to FP32, about 494 MB on the device. The T4 has no native BF16, so the repo’s `precision` returns plain FP32 there; on an H100 it uses BF16 autocast. Loading also checks that `tokenizer.json` hashes to the value recorded at training time.

**Sampling.** Generation feeds the most recent 256 tokens back through the model and reads the logits $z$ at the last position. Temperature $\tau$ rescales them, $p=\operatorname{softmax}(z/\tau)$. Nucleus (top‑$p$) sampling sorts tokens by probability and keeps the smallest prefix whose total probability reaches $p$; the code drops every token whose *preceding* cumulative probability is already at least $p$, then samples from the renormalized rest. $\tau=0$ means always taking the most likely token. The pad token is never sampled, and an end-of-text token stops generation.

The project’s settings are $\tau=0.7$, top‑$p=0.9$, 96 new tokens, seed 336, and the three fixed prompts in `PROMPTS`. Modal’s `sample` job uses the same function on CPU. Colab on CPU reproduces Modal’s samples exactly; on a T4 the random draws differ, so compare quality rather than exact text.

**Held-out loss.** The last cell recomputes validation loss on the run’s `valid.bin` with the repo’s `validate`: the same 512 evenly spaced windows the trainer uses. The H100 computed its value in BF16 and the T4 computes in FP32, so expect agreement to about two decimal places. [`generate`](https://github.com/xqzr2000/Stanford-CS336-Llama-Adaptation/blob/main/llama_lab/evaluate.py#L35) · [`load_for_eval`](https://github.com/xqzr2000/Stanford-CS336-Llama-Adaptation/blob/main/llama_lab/evaluate.py#L25) · [`export_weights`](https://github.com/xqzr2000/Stanford-CS336-Llama-Adaptation/blob/main/llama_lab/train.py#L207)

In [ ]:
final_model = final_tokenizer = None
if model_folder is None:
    print("Set MODEL_FOLDER in the files section to load the final model.")
else:
    weights_path = find_weights(model_folder)
    final_model, final_tokenizer, final_info = load_for_eval(weights_path, model_folder, device)
    final_model.eval()
    weights = sum(p.numel() for p in final_model.parameters())
    print(f"{weights_path.name}: step {final_info['step']:,}, {final_info['tokens']:,} tokens, "
          f"val_loss {final_info['val_loss']:.3f} (perplexity {np.exp(final_info['val_loss']):.1f})")
    print(f"{weights:,} parameters | {next(final_model.parameters()).dtype} on {device}")
    if device.type == "cuda":
        print(f"GPU memory allocated: {torch.cuda.memory_allocated(device) / 1e6:,.0f} MB")

In [ ]:
PROMPT = "Photosynthesis is the process by which plants"  # @param {"type":"string"}
TEMPERATURE = 0.7  # @param {"type":"number"}
TOP_P = 0.9  # @param {"type":"number"}
MAX_NEW = 96  # @param {"type":"integer"}

def continue_text(model, tokenizer, prompt, **settings):
    with attention_backend(device):
        return generate(model, tokenizer, prompt, device, **settings)

if final_model is None:
    print("Load the final model first.")
else:
    torch.manual_seed(SEED)
    settings = dict(max_new=MAX_NEW, temperature=TEMPERATURE, top_p=TOP_P)
    print(f"Project prompts | temperature {TEMPERATURE}, top-p {TOP_P}, seed {SEED}, {device}\n")
    for prompt in PROMPTS:
        print(f"▶ {prompt}\n{continue_text(final_model, final_tokenizer, prompt, **settings)}\n")
    print(f"▶ {PROMPT}  (your prompt)\n{continue_text(final_model, final_tokenizer, PROMPT, **settings)}\n")
    greedy = continue_text(final_model, final_tokenizer, PROMPT, max_new=MAX_NEW, temperature=0, top_p=1)
    print(f"▶ {PROMPT}  (greedy, temperature 0)\n{greedy}")

In [ ]:
valid_path = model_folder / "valid.bin" if model_folder else None
if final_model is None or valid_path is None or not valid_path.is_file():
    print("Needs the final model and valid.bin from the Colab folder.")
else:
    valid_tokens = np.fromfile(valid_path, dtype="<u2")
    with attention_backend(device):
        colab_loss = validate(final_model, valid_tokens, 64, device)
    final_model.eval()
    print(f"Validation loss on {device}: {colab_loss:.4f} | recorded at training time: {final_info['val_loss']:.4f}")
    print(f"Perplexity: {np.exp(colab_loss):.2f}")

## Chapter 5 · SST‑5 on the T4

SST‑5 labels movie-review sentences on five sentiment levels, from very negative (0) to very positive (4). It has 8,544 training and 1,101 development sentences.

**Classifier.** Each sentence is tokenized with the run’s tokenizer, cut to at most 127 tokens, and given an end-of-text token. The backbone returns features $H\in\mathbb{R}^{T\times D}$. Because attention is causal, the last real position $t^\ast$ has read the whole sentence, so the classifier uses

$$
\hat y=\operatorname{softmax}\big(W\,\operatorname{dropout}(H_{t^\ast})+b\big),\qquad W\in\mathbb{R}^{5\times D},
$$

and trains every weight, backbone included, with cross-entropy, AdamW at $3\times10^{-5}$, batch size 32 and gradient clipping at 1.0. Padding comes after $t^\ast$, so the causal mask keeps it out of the pooled features.

**Baseline.** Always predicting the most common training label scores the majority-class accuracy on the dev set. Report dev accuracy by epoch next to it.

**On the T4.** Full fine-tuning in FP32 needs about 2 GB for weights, gradients and AdamW moments, plus activations for 32 short sentences, well within 15 GB. Expect a few minutes per epoch.

`SST5_MODEL = "practice"` fine-tunes the small chapter‑3 model to check the pipeline on any device. `"final"` fine-tunes a copy of the run’s weights; it requires the T4 unless you allow CPU, which takes hours. Results are written to `sst5-colab-seed<seed>` in `MODEL_FOLDER`, so they persist in Drive. [`finetune_sst5`](https://github.com/xqzr2000/Stanford-CS336-Llama-Adaptation/blob/main/llama_lab/evaluate.py#L128) · [`SentimentClassifier`](https://github.com/xqzr2000/Stanford-CS336-Llama-Adaptation/blob/main/llama_lab/evaluate.py#L77)

In [ ]:
SST5_MODEL = "practice"  # @param ["practice", "final"]
EPOCHS = 3  # @param {"type":"integer"}
BATCH_SIZE = 32  # @param {"type":"integer"}
SST5_SEED = 336  # @param {"type":"integer"}
ALLOW_CPU = False  # @param {"type":"boolean"}

sst_root = workspace / "sst5-data"
if SST5_MODEL == "final":
    if model_folder is None:
        raise ValueError("Set MODEL_FOLDER to fine-tune the final model.")
    if device.type != "cuda" and not ALLOW_CPU:
        raise RuntimeError("Fine-tuning the 124M model on CPU takes hours. Use the T4 or set ALLOW_CPU.")
    source, tokenizer_root = find_weights(model_folder), model_folder
    sst_output = model_folder / f"sst5-colab-seed{SST5_SEED}"
else:
    source, tokenizer_root = practice_root / "latest.pt", slice_root
    sst_output = workspace / f"sst5-practice-seed{SST5_SEED}"

with attention_backend(device):
    finetune_sst5(source, tokenizer_root, sst_root, sst_output, device,
                  EPOCHS, BATCH_SIZE, SST5_SEED, commit=lambda: None)
sst_metrics = json.loads((sst_output / "metrics.json").read_text())
print(f"\nSource step {sst_metrics['step']:,} | seed {sst_metrics['seed']}")
print("Dev accuracy by epoch:", [round(score, 4) for score in sst_metrics["dev_accuracy_by_epoch"]])
print(f"Best {sst_metrics['best_dev_accuracy']:.4f} vs majority class {sst_metrics['majority_dev_accuracy']:.4f}")
print(f"Saved best.pt and metrics.json in {sst_output}")

## Results log

Fill this in as the project runs. Keep the run ID, step, and settings with every number.

| Chapter | Record |
| --- | --- |
| **2 · FineWeb‑Edu** | Prep app ID, manifest token counts, per-shard download and encoding times, any retries. Bytes per token for the run’s tokenizer. |
| **3 · Training** | Pilot throughput, chosen `--schedule-steps`, step‑0 and checkpoint validation loss, steady tokens/s, verified resume, and billed cost. |
| **4 · Final model** | Exported step, validation loss on Colab vs training time, and samples for the three prompts with their settings. |
| **5 · SST‑5** | Source step, seed, device, dev accuracy by epoch, and majority-class baseline. Repeat with a second seed if time allows. |

**My notes** · What did I expect before running each chapter? Which tensor shape or equation would I like to revisit after seeing the real checkpoint?